# PyG molecular regression (notebooks)

Copy any **code** cell into another notebook, or run this file from the repo with the kernel’s working directory set to the **project root** (folder that contains `src/` and `pyproject.toml`).

Requires: `pip install openadnet[dl]`.

## 1. Put `src` on `sys.path` (paste at top of a notebook)

Run once per session if imports like `models` fail.

In [1]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
if (_root / "src").is_dir():
    sys.path.insert(0, str(_root / "src"))
else:
    raise RuntimeError("Start the kernel cwd from the openadnet repo (or folder containing pyproject.toml)")

## 2. K-fold CV (full data)

Same idea as `scripts/cv_gnn_regressor.py`. Change `architecture` to `gcn`, `gat`, `mpnn`, `graphconv`, or `attentivefp`.

In [3]:
from IPython.display import display

from baseline import BaselineCVConfig
from load_data import train
from models.cv_dl import run_gnn_regressor_cv

cfg = BaselineCVConfig(n_splits=3, shuffle=True, cv_random_state=0, y_col="pEC50")
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="gin",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=3,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

gnn_cv:   0%|          | 0/3 [00:00<?, ?fold/s]

,fold,rmse,mae,r2,n_train,n_val
0,0,0.817705,0.643558,0.437740,2760,1380
1,1,0.823294,0.623762,0.464761,2760,1380
2,2,0.896853,0.667897,0.388807,2760,1380


mean_rmse    0.845951
std_rmse     0.036066
mean_mae     0.645072
std_mae      0.018050
mean_r2      0.430436
std_r2       0.031435
dtype: float64

In [ ]:
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="gat",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

gnn_cv:   0%|          | 0/3 [00:00<?, ?fold/s]

## 3. K-fold CV (small subset — fast)

Mirrors `examples/quick_cv_gnn_subset.py`.

In [ ]:
from baseline import BaselineCVConfig
from load_data import train
from models.cv_dl import run_gnn_regressor_cv

small = train.head(300).copy()
cfg = BaselineCVConfig(n_splits=3, shuffle=True, cv_random_state=42, y_col="pEC50")
fold_df, summary = run_gnn_regressor_cv(
    small,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="mpnn",
    config=cfg,
    epochs=2,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=48,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
)

## 4. Single train / validation split (holdout)

Same idea as `examples/holdout_regression.py --backend gnn`.

In [ ]:
from load_data import train
from models.cv_dl import prepare_regression_frame, regression_metrics
from models.data import graph_regression_from_dataframe
from models.data.graph import train_val_split_graph
from models.nn.pyg_regressor import PyGMoleculeRegressor

work = prepare_regression_frame(train, "SMILES", ["pEC50"])
full_ds = graph_regression_from_dataframe(work, "SMILES", ["pEC50"])
tr_ds, va_ds = train_val_split_graph(full_ds, val_fraction=0.15, random_state=0)

model = PyGMoleculeRegressor(
    n_tasks=1,
    architecture="gcn",
    hidden_dim=64,
    num_layers=3,
    gat_heads=4,
)
model.fit(
    tr_ds,
    epochs=2,
    batch_size=32,
    learning_rate=1e-3,
    val_dataset=va_ds,
    show_progress=True,
)
pred = model.predict(va_ds, show_progress=False)
regression_metrics(va_ds.y, pred)

## 5. `GNNRegressor` = GIN-only alias

Drop-in if you want the old default without passing `architecture`.

In [ ]:
from models import GNNRegressor

model = GNNRegressor(n_tasks=1, hidden_dim=64, num_layers=3)
# same as PyGMoleculeRegressor(..., architecture="gin")

## 6. Architecture names (for `architecture=`)

`gat` and `attentivefp` use `gat_heads`.

In [ ]:
from models.nn.registry import ARCHITECTURES

list(ARCHITECTURES)

## 7. Optional: force CPU

Use if CUDA errors in notebooks.

In [ ]:
import os
import torch

os.environ.setdefault("OPENADNET_FORCE_CPU", "1")
device = torch.device("cpu")
# PyGMoleculeRegressor(..., device=device)